In [2]:
from kloppy import sportec, statsbomb
from kloppy.domain.models import PitchDimensions, Dimension
from databallpy import get_game_from_kloppy

print("Parsing local XML tracking and JSON event files via Kloppy...")

# 1. Load Local Sportec Tracking Data
tracking_dataset = sportec.load_tracking(
    raw_data="hdfc_tracking_event_matches/tracking/positional_data_raw/MLS-COM-000001_MLS-SEA-0001KA_MLS-MAT-0009B7.xml",
    meta_data="hdfc_tracking_event_matches/tracking/match_information/MLS-COM-000001_MLS-SEA-0001KA_MLS-MAT-0009B7.xml",
    coordinates="sportec" 
)

# 3. Load Local StatsBomb Event Data 
event_dataset = statsbomb.load(
    event_data="hdfc_tracking_event_matches/events/4037572.json",
    lineup_data="hdfc_tracking_event_matches/lineups/4037572.json"
)

# --- FIX: Define an identical custom pitch dimensions structure for both ---
# Rather than manually creating PitchDimensions, we can copy the dimensions 
# from the StatsBomb dataset OR force both to use StatsBomb's exact coordinate system.
# The safest approach for databallpy is transforming BOTH to ensure metadata parity.

tracking_dataset = tracking_dataset.transform(to_coordinate_system="statsbomb")
event_dataset = event_dataset.transform(to_coordinate_system="statsbomb")

# Explicitly align the pitch metadata attributes so databallpy's strict check passes
tracking_dataset.metadata.pitch_dimensions = event_dataset.metadata.pitch_dimensions

print("Rescaled both datasets to identical coordinate systems and metadata.")

# 4. Transform and combine into a DataBallPy Game object
game = get_game_from_kloppy(
    tracking_dataset=tracking_dataset, 
    event_dataset=event_dataset
)

print("\n--- MATCH COMBINATION COMPLETED ---")
print(f"Tracking Dataset Frames: {len(game.tracking_data)}")
print(f"Event Dataset Logs: {len(game.event_data)}")

Parsing local XML tracking and JSON event files via Kloppy...
Rescaled both datasets to identical coordinate systems and metadata.


/Users/mbasurto/Documents/Projects/databallpy/databallpy/utils/get_game.py:899: UserWarning: Game dates in kloppy TrackingDataset and EventDataset are not equal. Setting both to pd.Timestamp('1975-01-01').
  warnings.warn(



--- MATCH COMBINATION COMPLETED ---
Tracking Dataset Frames: 166538
Event Dataset Logs: 3455


In [3]:
# Sync the event and tracking data
game.synchronise_tracking_and_event_data()
# Combined data can be accessed via game.tracking_data and game.event_data DataFrames
print("\n--- SYNCHRONIZATION COMPLETED ---")
# Overall sync certainty
print(f"Sync Certainty: {game.tracking_data['sync_certainty'].mean():.4f}")
# Print certainty variance to check for consistency
print(f"Sync Certainty Variance: {game.tracking_data['sync_certainty'].var():.6f}")


--- SYNCHRONIZATION COMPLETED ---
Sync Certainty: 0.9444
Sync Certainty Variance: 0.010923


In [4]:
import itertools
import numpy as np
import pandas as pd
# Import the module where the function lives
import databallpy.utils.synchronise_tracking_and_event_data as sync_module 

# Save a reference to the original function so we don't break anything permanently
original_combine_cost_functions = sync_module.combine_cost_functions

# --- STEP 1: Define your Grid Search Space ---
# Adjust these lists to the specific ranges you want to test
grid_space = {
    "time_cost": [0.25, 0.5, 1.0, 1.5, 2.0],
    "ball_event_dist_cost": [1.0, 1.5, 2.0],
    "ball_player_dist_cost": [0.5, 1.0, 1.5, 2.0],
    "ball_acc_cost": [1.0],
    "player_ball_dist_inc_cost": [1.0, 2.0],
    "goal_angle_cost": [1.0]
}

# Generate all combinations
keys, values = zip(*grid_space.items())
experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]

print(f"Total grid search iterations to run: {len(experiments)}")

# --- STEP 2: Run the Grid Search ---
results = []

for i, custom_weights in enumerate(experiments):
    print(f"Running iteration {i+1}/{len(experiments)} with weights: {custom_weights}")
    
    # Define a wrapper function that forces your custom weights
    def mocked_combine_cost_functions(costs, keys, weights_dict=None):
        # We completely ignore the internal default and inject our grid-search weights
        return original_combine_cost_functions(costs, keys, weights_dict=custom_weights)
    
    # Monkey-patch the module to use our wrapped function
    sync_module.combine_cost_functions = mocked_combine_cost_functions
    
    try:
        # Run the sync process normally - it will now use your custom weights!
        game.synchronise_tracking_and_event_data()
        
        # Calculate your performance metrics (e.g., mean certainty)
        mean_certainty = game.tracking_data['sync_certainty'].mean()
        variance_certainty = game.tracking_data['sync_certainty'].var()
        
        # Store results
        result_entry = custom_weights.copy()
        result_entry["mean_certainty"] = mean_certainty
        result_entry["variance_certainty"] = variance_certainty
        results.append(result_entry)
        
    except Exception as e:
        print(f"Iteration {i+1} failed with error: {e}")
        
# --- STEP 3: Clean up and Restore original behavior ---
sync_module.combine_cost_functions = original_combine_cost_functions
print("\n--- GRID SEARCH COMPLETED & ORIGINAL FUNCTION RESTORED ---")

# Convert results to a DataFrame for easy sorting
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by="mean_certainty", ascending=False)
print(df_results.head())

Total grid search iterations to run: 120
Running iteration 1/120 with weights: {'time_cost': 0.25, 'ball_event_dist_cost': 1.0, 'ball_player_dist_cost': 0.5, 'ball_acc_cost': 1.0, 'player_ball_dist_inc_cost': 1.0, 'goal_angle_cost': 1.0}


Running iteration 2/120 with weights: {'time_cost': 0.25, 'ball_event_dist_cost': 1.0, 'ball_player_dist_cost': 0.5, 'ball_acc_cost': 1.0, 'player_ball_dist_inc_cost': 2.0, 'goal_angle_cost': 1.0}


Running iteration 3/120 with weights: {'time_cost': 0.25, 'ball_event_dist_cost': 1.0, 'ball_player_dist_cost': 1.0, 'ball_acc_cost': 1.0, 'player_ball_dist_inc_cost': 1.0, 'goal_angle_cost': 1.0}


Running iteration 4/120 with weights: {'time_cost': 0.25, 'ball_event_dist_cost': 1.0, 'ball_player_dist_cost': 1.0, 'ball_acc_cost': 1.0, 'player_ball_dist_inc_cost': 2.0, 'goal_angle_cost': 1.0}


Running iteration 5/120 with weights: {'time_cost': 0.25, 'ball_event_dist_cost': 1.0, 'ball_player_dist_cost': 1.5, 'ball_acc_cost': 1.0, 'player_ball_dist_inc_cost': 1.0, 'goal_angle_cost': 1.0}


...

Running iteration 117/120 with weights: {'time_cost': 2.0, 'ball_event_dist_cost': 2.0, 'ball_player_dist_cost': 1.5, 'ball_acc_cost': 1.0, 'player_ball_dist_inc_cost': 1.0, 'goal_angle_cost': 1.0}


Running iteration 118/120 with weights: {'time_cost': 2.0, 'ball_event_dist_cost': 2.0, 'ball_player_dist_cost': 1.5, 'ball_acc_cost': 1.0, 'player_ball_dist_inc_cost': 2.0, 'goal_angle_cost': 1.0}


Running iteration 119/120 with weights: {'time_cost': 2.0, 'ball_event_dist_cost': 2.0, 'ball_player_dist_cost': 2.0, 'ball_acc_cost': 1.0, 'player_ball_dist_inc_cost': 1.0, 'goal_angle_cost': 1.0}


Running iteration 120/120 with weights: {'time_cost': 2.0, 'ball_event_dist_cost': 2.0, 'ball_player_dist_cost': 2.0, 'ball_acc_cost': 1.0, 'player_ball_dist_inc_cost': 2.0, 'goal_angle_cost': 1.0}



--- GRID SEARCH COMPLETED & ORIGINAL FUNCTION RESTORED ---
    time_cost  ball_event_dist_cost  ball_player_dist_cost  ball_acc_cost  \
6        0.25                   1.0                    2.0            1.0   
7        0.25                   1.0                    2.0            1.0   
30       0.50                   1.0                    2.0            1.0   
4        0.25                   1.0                    1.5            1.0   
14       0.25                   1.5                    2.0            1.0   

    player_ball_dist_inc_cost  goal_angle_cost  mean_certainty  \
6                         1.0              1.0        0.951246   
7                         2.0              1.0        0.950245   
30                        1.0              1.0        0.949984   
4                         1.0              1.0        0.949918   
14                        1.0              1.0        0.949662   

    variance_certainty  
6             0.014949  
7             0.013741  
30   

In [ ]:
pd.set_option('display.max_columns', None)
df_results.sort_values(by = 'variance_certainty', ascending=True).head(30)

,time_cost,ball_event_dist_cost,ball_player_dist_cost,ball_acc_cost,player_ball_dist_inc_cost,goal_angle_cost,mean_certainty,variance_certainty
97,2.0,1.0,0.5,1.0,2.0,1.0,0.943500,0.005241
96,2.0,1.0,0.5,1.0,1.0,1.0,0.942937,0.005581
99,2.0,1.0,1.0,1.0,2.0,1.0,0.944096,0.006059
73,1.5,1.0,0.5,1.0,2.0,1.0,0.943958,0.006190
98,2.0,1.0,1.0,1.0,1.0,1.0,0.943632,0.006489
72,1.5,1.0,0.5,1.0,1.0,1.0,0.943485,0.006642
101,2.0,1.0,1.5,1.0,2.0,1.0,0.945083,0.006931
105,2.0,1.5,0.5,1.0,2.0,1.0,0.942138,0.006949
75,1.5,1.0,1.0,1.0,2.0,1.0,0.944676,0.007080
104,2.0,1.5,0.5,1.0,1.0,1.0,0.942155,0.007335
